In [1]:
!pip uninstall torch

In [3]:
# !pip freeze > req.txt
!pip install torch==2.8.0 torchvision==0.23.0 torchaudio==2.8.0 --index-url https://download.pytorch.org/whl/cu129

^C


Defaulting to user installation because normal site-packages is not writeable
Looking in indexes: https://download.pytorch.org/whl/cu129
   ---------------------------------------- 0.0/3.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/3.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/3.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/3.6 GB ? eta -:--:--
   ---------------------------------------- 0.0/3.6 GB 1.2 MB/s eta 0:49:15
   ---------------------------------------- 0.0/3.6 GB 1.7 MB/s eta 0:34:36
   ---------------------------------------- 0.0/3.6 GB 2.7 MB/s eta 0:22:22
   ---------------------------------------- 0.0/3.6 GB 3.9 MB/s eta 0:15:17
   ---------------------------------------- 0.0/3.6 GB 6.0 MB/s eta 0:09:51
   ---------------------------------------- 0.0/3.6 GB 9.7 MB/s eta 0:06:08
   ---------------------------------------- 0.0/3.6 GB 11.7 MB/s eta 0:05:04
   ---------------------------------------- 0.0/3.6 GB

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
autogluon-multimodal 1.6.1 requires torch<2.14,>=2.10, but you have torch 2.8.0+cu129 which is incompatible.
autogluon-timeseries 1.6.1 requires torch<2.14,>=2.10, but you have torch 2.8.0+cu129 which is incompatible.


In [1]:
import pandas as pd
df_train=pd.read_csv('data/train.csv')
df_test=pd.read_csv('data/test.csv')
df_train_merged = pd.read_csv('data/train_merged.csv')

df_train_merged.head()

,Driver,Compound,Race,Year,PitStop,LapNumber,Stint,TyreLife,Position,LapTime (s),LapTime_Delta,Cumulative_Degradation,RaceProgress,Position_Change,PitNextLap
0,SWI,MEDIUM,Mexico City Grand Prix,2023,0,6,1,6.0,12,83.921,-21.244,-10.320,0.084507,0.0,0.0
1,TRU,HARD,Italian Grand Prix,2024,0,24,2,17.0,15,83.845,-22.913,-33.696,0.311688,-9.0,1.0
2,TSU,MEDIUM,Monaco Grand Prix,2023,0,23,1,23.0,9,79.239,0.087,-12.078,0.302632,0.0,0.0
3,PEA,HARD,Italian Grand Prix,2022,1,50,2,33.0,11,87.076,-13.929,-31.804,0.694444,3.0,1.0
4,ANT,HARD,Monaco Grand Prix,2025,0,49,1,49.0,12,78.328,-0.516,-33.315,0.653333,0.0,0.0


In [2]:
from autogluon.tabular import TabularDataset, TabularPredictor

In [3]:
TARGET = 'PitNextLap'

print(df_train[TARGET].value_counts())
print(df_train_merged[TARGET].value_counts())

PitNextLap
0.0    351759
1.0     87381
Name: count, dtype: int64
PitNextLap
0.0    427273
1.0    113172
Name: count, dtype: int64


In [4]:
# predictor = TabularPredictor(label=TARGET,eval_metric='roc_auc').fit(
#     train_data=df_train,
#     ag_args_fit={"num_gpus": 2},
#     time_limit=3600*9,
#     presets='best_quality',
#     verbosity=3,
#     num_stack_levels=0
# )

In [5]:
import warnings
warnings.filterwarnings(
    "ignore", category=FutureWarning, message=".*downcast.*"
)
_orig_fillna = pd.DataFrame.fillna
def _patched_fillna(self, *args, **kwargs):
    kwargs.pop("downcast", None)
    return _orig_fillna(self, *args, **kwargs)

In [6]:
models = {
    "GBM": [
        {},  # Generates LightGBM_BAG_L1
        {"extra_trees": True, "ag_args": {"name_suffix": "Large"}},  # Generates LightGBMLarge_BAG_L1
    ],
    "XGB": {},  # Generates XGBoost_BAG_L1
    "CAT": {},  # Generates CatBoost_BAG_L1
}


In [7]:
predictor = TabularPredictor(label=TARGET,eval_metric='roc_auc', path='ag_models5').fit(
    train_data=df_train_merged,
    ag_args_fit={"num_gpus": 1},
    time_limit=3600*9,
    presets='best_quality',
    verbosity=3,
    num_stack_levels=0,
    hyperparameters=models,
)

Verbosity: 3 (Detailed Logging)
=================== System Info ===================
AutoGluon Version:  1.6.1
Python Version:     3.13.13
Operating System:   Windows
Platform Machine:   AMD64
Platform Version:   10.0.26200
CPU Count:          16
Pytorch Version:    2.8.0+cu129
CUDA Version:       12.9
GPU Memory:         GPU 0: 15.93/15.93 GB
Total GPU Memory:   Free: 15.93 GB, Allocated: 0.00 GB, Total: 15.93 GB
GPU Count:          1
Memory Avail:       4.85 GB / 15.06 GB (32.2%)
Disk Space Avail:   717.41 GB / 930.47 GB (77.1%)
Presets specified: ['best_quality']
============ fit kwarg info ============
User Specified kwargs:
{'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 'num_stack_levels': 0,
 'verbosity': 3}
Full kwargs:
{'_experimental_dynamic_hyperparameters': False,
 '_feature_generator_kwargs': None,
 '_save_bag_folds': None,
 'adapt_num_bag_folds_to_n_classes': False,
 'ag_args': None,
 'ag_args_ensemble': None,
 'ag_args_fit': {'num_gpus': 1},
 'auto_stack': True,
 '

[50]	valid_set's binary_logloss: 0.281445
[100]	valid_set's binary_logloss: 0.260036
[150]	valid_set's binary_logloss: 0.25199
[200]	valid_set's binary_logloss: 0.247187
[250]	valid_set's binary_logloss: 0.243627
[300]	valid_set's binary_logloss: 0.240783
[350]	valid_set's binary_logloss: 0.23862
[400]	valid_set's binary_logloss: 0.236582
[450]	valid_set's binary_logloss: 0.234896
[500]	valid_set's binary_logloss: 0.233701
[550]	valid_set's binary_logloss: 0.232697
[600]	valid_set's binary_logloss: 0.231964
[650]	valid_set's binary_logloss: 0.231146
[700]	valid_set's binary_logloss: 0.230417
[750]	valid_set's binary_logloss: 0.22974
[800]	valid_set's binary_logloss: 0.229188
[850]	valid_set's binary_logloss: 0.228568
[900]	valid_set's binary_logloss: 0.228074
[950]	valid_set's binary_logloss: 0.227555
[1000]	valid_set's binary_logloss: 0.227149
[1050]	valid_set's binary_logloss: 0.226735
[1100]	valid_set's binary_logloss: 0.226487
[1150]	valid_set's binary_logloss: 0.226146
[1200]	vali

	Fit constraints were validated upstream on the full training data, skipping...
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'seed': 0, 'device': 'gpu'}


[50]	valid_set's binary_logloss: 0.283034
[100]	valid_set's binary_logloss: 0.261641
[150]	valid_set's binary_logloss: 0.25394
[200]	valid_set's binary_logloss: 0.248926
[250]	valid_set's binary_logloss: 0.245444
[300]	valid_set's binary_logloss: 0.242644
[350]	valid_set's binary_logloss: 0.240329
[400]	valid_set's binary_logloss: 0.238109
[450]	valid_set's binary_logloss: 0.236788
[500]	valid_set's binary_logloss: 0.23558
[550]	valid_set's binary_logloss: 0.234712
[600]	valid_set's binary_logloss: 0.233943
[650]	valid_set's binary_logloss: 0.233117
[700]	valid_set's binary_logloss: 0.232533
[750]	valid_set's binary_logloss: 0.231818
[800]	valid_set's binary_logloss: 0.231282
[850]	valid_set's binary_logloss: 0.230866
[900]	valid_set's binary_logloss: 0.230417
[950]	valid_set's binary_logloss: 0.230038
[1000]	valid_set's binary_logloss: 0.2298
[1050]	valid_set's binary_logloss: 0.229532
[1100]	valid_set's binary_logloss: 0.229189
[1150]	valid_set's binary_logloss: 0.228974
[1200]	valid

	Fit constraints were validated upstream on the full training data, skipping...
	Fitting 10000 rounds... Hyperparameters: {'learning_rate': 0.05, 'seed': 0, 'device': 'gpu'}


...

	Fit constraints were validated upstream on the full training data, skipping...


[0]	validation_0-logloss:0.47348
[50]	validation_0-logloss:0.26290
[100]	validation_0-logloss:0.24852
[150]	validation_0-logloss:0.24127
[200]	validation_0-logloss:0.23612
[250]	validation_0-logloss:0.23289
[300]	validation_0-logloss:0.22953
[350]	validation_0-logloss:0.22705
[400]	validation_0-logloss:0.22519
[450]	validation_0-logloss:0.22390
[500]	validation_0-logloss:0.22231
[550]	validation_0-logloss:0.22131
[600]	validation_0-logloss:0.22001
[650]	validation_0-logloss:0.21867
[700]	validation_0-logloss:0.21780
[750]	validation_0-logloss:0.21690
[800]	validation_0-logloss:0.21625
[850]	validation_0-logloss:0.21585
[900]	validation_0-logloss:0.21539
[950]	validation_0-logloss:0.21489
[1000]	validation_0-logloss:0.21453
[1050]	validation_0-logloss:0.21423
[1100]	validation_0-logloss:0.21379
[1150]	validation_0-logloss:0.21356
[1200]	validation_0-logloss:0.21316
[1250]	validation_0-logloss:0.21281
[1300]	validation_0-logloss:0.21248
[1350]	validation_0-logloss:0.21220
[1400]	validati

	Fit constraints were validated upstream on the full training data, skipping...


[0]	validation_0-logloss:0.47342
[50]	validation_0-logloss:0.26227
[100]	validation_0-logloss:0.24766
[150]	validation_0-logloss:0.24057
[200]	validation_0-logloss:0.23543
[250]	validation_0-logloss:0.23195
[300]	validation_0-logloss:0.22857
[350]	validation_0-logloss:0.22593
[400]	validation_0-logloss:0.22427
[450]	validation_0-logloss:0.22256
[500]	validation_0-logloss:0.22103
[550]	validation_0-logloss:0.21953
[600]	validation_0-logloss:0.21839
[650]	validation_0-logloss:0.21758
[700]	validation_0-logloss:0.21653
[750]	validation_0-logloss:0.21572
[800]	validation_0-logloss:0.21481
[850]	validation_0-logloss:0.21410
[900]	validation_0-logloss:0.21361
[950]	validation_0-logloss:0.21309
[1000]	validation_0-logloss:0.21254
[1050]	validation_0-logloss:0.21210
[1100]	validation_0-logloss:0.21190
[1150]	validation_0-logloss:0.21159
[1200]	validation_0-logloss:0.21124
[1250]	validation_0-logloss:0.21088
[1300]	validation_0-logloss:0.21071
[1350]	validation_0-logloss:0.21047
[1400]	validati

Saving c:\Darshak\Projects\Hackathon\ag_models5\models\XGBoost_BAG_L1\utils\oof.pkl
Saving c:\Darshak\Projects\Hackathon\ag_models5\models\XGBoost_BAG_L1\model.pkl
	0.9582	 = Validation score   (roc_auc)
	558.77s	 = Training   runtime
	4.33s	 = Validation runtime
	15612.6	 = Inference  throughput (rows/s | 67556 batch size)
Saving c:\Darshak\Projects\Hackathon\ag_models5\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\LightGBM_BAG_L1\utils\oof.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\LightGBMLarge_BAG_L1\utils\oof.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\CatBoost_BAG_L1\utils\oof.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\XGBoost_BAG_L1\utils\oof.pkl
Model configs that will be trained (in order):
	WeightedEnsemble_L2: 	{'ag_args': {'problem_types': ['binary', 'multiclass', 'regression', 'quantile', 'softclass'], 'valid_base': False, 'name_bag_suffix': '', 'model_type': <class 'autogluon.core.models.gr

In [8]:
predictor.leaderboard()

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,0.959994,roc_auc,22.646903,1704.332377,0.049652,5.011189,2,True,5
1,XGBoost_BAG_L1,0.958240,roc_auc,4.327017,558.767684,4.327017,558.767684,1,True,4
2,CatBoost_BAG_L1,0.958136,roc_auc,4.220672,966.620621,4.220672,966.620621,1,True,3
3,LightGBMLarge_BAG_L1,0.952897,roc_auc,14.049562,173.932884,14.049562,173.932884,1,True,2
4,LightGBM_BAG_L1,0.951028,roc_auc,5.040709,87.416877,5.040709,87.416877,1,True,1


In [10]:
df=predictor.predict_proba(df_test)
df.head()

Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\CatBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\LightGBMLarge_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\WeightedEnsemble_L2\model.pkl


,0,1
0,0.994529,0.005471
1,0.994558,0.005442
2,0.995153,0.004847
3,0.828029,0.171971
4,0.073202,0.926798


In [ ]:
# import os
# import pandas as pd
# from autogluon.tabular import TabularPredictor

# # ============================================================
# # CONFIG
# # ============================================================

# MODEL_DIR = "./ag_models2"

# all_leaderboards = []

# # ============================================================
# # SCAN & LOAD ALL TRAINED PREDICTORS
# # ============================================================

# if os.path.exists(MODEL_DIR):
#     # Check both subdirectories and root directory for saved predictors
#     folder_candidates = [MODEL_DIR] + [
#         os.path.join(MODEL_DIR, d) 
#         for d in os.listdir(MODEL_DIR) 
#         if os.path.isdir(os.path.join(MODEL_DIR, d))
#     ]

#     for folder_path in folder_candidates:
#         # Check if directory contains a valid predictor file
#         if os.path.exists(os.path.join(folder_path, "predictor.pkl")):
#             try:
#                 folder_name = os.path.basename(folder_path)
#                 print(f"Loading predictor from: {folder_path}")
                
#                 # Load predictor from disk
#                 predictor = TabularPredictor.load(folder_path)
                
#                 # Fetch leaderboard
#                 lb = predictor.leaderboard(silent=True)
#                 lb.insert(0, "folder_name", folder_name)
                
#                 all_leaderboards.append(lb)
#             except Exception as e:
#                 print(f"Failed to load predictor from {folder_path}: {e}")

# # ============================================================
# # MERGE AND PRESENT COMBINED LEADERBOARD
# # ============================================================

# if all_leaderboards:
#     combined_leaderboard = pd.concat(all_leaderboards, ignore_index=True)

#     # Sort all trained models by validation ROC-AUC score descending
#     combined_leaderboard = combined_leaderboard.sort_values(
#         by="score_val", ascending=False
#     ).reset_index(drop=True)

#     print("\n" + "=" * 80)
#     print("COMBINED LEADERBOARD OF ALL LOADED MODELS")
#     print("=" * 80)

#     display(combined_leaderboard)
# else:
#     print(f"No valid AutoGluon predictors (`predictor.pkl`) found under '{MODEL_DIR}'.")

Loading predictor from: ./ag_models2\CAT


Loading: c:\Darshak\Projects\Hackathon\ag_models2\CAT\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\CAT\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\CAT\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\GBM\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\GBM\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\GBM\models\trainer.pkl


Loading predictor from: ./ag_models2\GBM
Loading predictor from: ./ag_models2\RF


Loading: c:\Darshak\Projects\Hackathon\ag_models2\RF\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\RF\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\RF\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\trainer.pkl


Loading predictor from: ./ag_models2\XGB

COMBINED LEADERBOARD OF ALL LOADED MODELS


,folder_name,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,XGB,WeightedEnsemble_L2,0.958240,roc_auc,4.397609,601.833442,0.053591,0.057095,2.0,1.0,2.0
1,XGB,XGBoost_BAG_L1,0.958240,roc_auc,4.344018,601.776347,4.344018,601.776347,1.0,1.0,1.0
2,GBM,WeightedEnsemble_L2,0.953617,roc_auc,18.501622,255.991845,0.060542,2.829900,2.0,1.0,3.0
3,GBM,LightGBMLarge_BAG_L1,0.952897,roc_auc,13.476475,172.914947,13.476475,172.914947,1.0,1.0,2.0
4,GBM,LightGBM_BAG_L1,0.951028,roc_auc,4.964605,80.246999,4.964605,80.246999,1.0,1.0,1.0
5,CAT,WeightedEnsemble_L2,0.950199,roc_auc,0.780571,179.753519,0.057653,0.067708,2.0,1.0,2.0
6,CAT,CatBoost_BAG_L1,0.950199,roc_auc,0.722918,179.685812,0.722918,179.685812,1.0,1.0,1.0


In [ ]:
# predictor = TabularPredictor.load("ag_models2/XGB")
# predictor.info()

Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\predictor.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\learner.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\trainer.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F3\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F4\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F5\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F6\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\S1F7\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_B

{'path': 'c:\\Darshak\\Projects\\Hackathon\\ag_models2\\XGB',
 'label': 'PitNextLap',
 'random_state': 0,
 'version': '1.6.1',
 'features': ['Driver',
  'Compound',
  'Race',
  'Year',
  'PitStop',
  'LapNumber',
  'Stint',
  'TyreLife',
  'Position',
  'LapTime (s)',
  'LapTime_Delta',
  'Cumulative_Degradation',
  'RaceProgress',
  'Position_Change'],
 'feature_metadata_in': <autogluon.common.features.feature_metadata.FeatureMetadata at 0x18a176d61b0>,
 'time_fit_preprocessing': 0.8181874752044678,
 'time_fit_training': 607.6769812107086,
 'time_fit_total': 608.4951686859131,
 'time_limit': 1800,
 'time_train_start': 1786370652.9010363,
 'num_rows_train': 540445,
 'num_cols_train': 14,
 'num_rows_val': None,
 'num_rows_test': None,
 'num_classes': 2,
 'problem_type': 'binary',
 'eval_metric': 'roc_auc',
 'best_model': 'WeightedEnsemble_L2',
 'best_model_score_val': np.float64(0.9582403341215795),
 'best_model_stack_level': 2,
 'num_models_trained': 2,
 'num_bag_folds': 8,
 'max_stack

In [76]:
df=predictor.predict_proba(df_test)
df.head()

Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models2\XGB\models\WeightedEnsemble_L2\model.pkl


,0,1
0,0.995741,0.004259
1,0.995626,0.004374
2,0.996494,0.003506
3,0.800449,0.199551
4,0.039684,0.960316


In [11]:
# df.to_csv("With_driver_feature_submission.csv")
df_sample_out=pd.read_csv('data/sample_submission.csv')
df_sample_out.head()

,id,PitNextLap
0,439140,0
1,439141,0
2,439142,0
3,439143,0
4,439144,0


In [12]:
df_sample_out['PitNextLap']=df[1]

In [13]:
df_sample_out.head()

,id,PitNextLap
0,439140,0.005471
1,439141,0.005442
2,439142,0.004847
3,439143,0.171971
4,439144,0.926798


In [14]:
df_sample_out.to_csv("My_output/all_model_together_submission.csv")

Loading: c:\Darshak\Projects\Hackathon\ag_models2\CAT\models\CatBoost_BAG_L1\model.pkl


Loading: c:\Darshak\Projects\Hackathon\ag_models2\CAT\models\WeightedEnsemble_L2\model.pkl


      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9126
Mismatches                         : 874
Accuracy (%)                       : 91.26
True Positives (Actual 1, Pred 1)  : 1527
True Negatives (Actual 0, Pred 0)  : 7599
False Positives (Actual 0, Pred 1) : 401
False Negatives (Actual 1, Pred 0) : 473
Loaded: sample_part_1_10000k.csv -> Shape: (10000, 15)


In [15]:
from multi_sampling_test_predictor import process_and_evaluate_all_csvs
data_dict = process_and_evaluate_all_csvs(predictor,folder_path="Sampling_data_to_test",drop_cols=[])

Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\CatBoost_BAG_L1\model.pkl



----------------------------------------
 Processing: sample_part_1_10000k.csv
----------------------------------------
ground_truth
0    8000
1    2000
Name: count, dtype: int64


Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\LightGBMLarge_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\WeightedEnsemble_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\CatBoost_BAG_L1\model.pkl


--- Per-File Analysis [sample_part_1_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9396
Mismatches                         : 604
Accuracy (%)                       : 93.96
True Positives (Actual 1, Pred 1)  : 1676
True Negatives (Actual 0, Pred 0)  : 7720
False Positives (Actual 0, Pred 1) : 280
False Negatives (Actual 1, Pred 0) : 324

----------------------------------------
 Processing: sample_part_2_10000k.csv
----------------------------------------
ground_truth
0    8500
1    1500
Name: count, dtype: int64


Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\LightGBMLarge_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\WeightedEnsemble_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\CatBoost_BAG_L1\model.pkl


...

--- Per-File Analysis [sample_part_3_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9360
Mismatches                         : 640
Accuracy (%)                       : 93.6
True Positives (Actual 1, Pred 1)  : 2118
True Negatives (Actual 0, Pred 0)  : 7242
False Positives (Actual 0, Pred 1) : 258
False Negatives (Actual 1, Pred 0) : 382

----------------------------------------
 Processing: sample_part_4_10000k.csv
----------------------------------------
ground_truth
0    7000
1    3000
Name: count, dtype: int64


Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\LightGBMLarge_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\WeightedEnsemble_L2\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\CatBoost_BAG_L1\model.pkl


--- Per-File Analysis [sample_part_4_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9335
Mismatches                         : 665
Accuracy (%)                       : 93.35
True Positives (Actual 1, Pred 1)  : 2566
True Negatives (Actual 0, Pred 0)  : 6769
False Positives (Actual 0, Pred 1) : 231
False Negatives (Actual 1, Pred 0) : 434

----------------------------------------
 Processing: sample_part_5_10000k.csv
----------------------------------------
ground_truth
0    9000
1    1000
Name: count, dtype: int64


Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\LightGBMLarge_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\XGBoost_BAG_L1\model.pkl
Loading: c:\Darshak\Projects\Hackathon\ag_models5\models\WeightedEnsemble_L2\model.pkl


--- Per-File Analysis [sample_part_5_10000k.csv] ---
      PREDICTION ANALYSIS REPORT        
Total Records                      : 10000
Correct Matches                    : 9516
Mismatches                         : 484
Accuracy (%)                       : 95.16
True Positives (Actual 1, Pred 1)  : 831
True Negatives (Actual 0, Pred 0)  : 8685
False Positives (Actual 0, Pred 1) : 315
False Negatives (Actual 1, Pred 0) : 169

      OVERALL CUMULATIVE ANALYSIS REPORT        
      PREDICTION ANALYSIS REPORT        
Total Records                      : 50000
Correct Matches                    : 47067
Mismatches                         : 2933
Accuracy (%)                       : 94.13
True Positives (Actual 1, Pred 1)  : 8469
True Negatives (Actual 0, Pred 0)  : 38598
False Positives (Actual 0, Pred 1) : 1402
False Negatives (Actual 1, Pred 0) : 1531
